# AgroSense — Pakistan Crop Stress Classification
## Simplified Interactive Walkthrough

> **For exact manuscript results, run `python src/train_models.py`.**
> This notebook demonstrates the methodology using a simplified grouped-CV workflow
> (non-nested single-pass CV on the training partition, fixed hyperparameters).
> It does **not** reproduce the full 5-fold outer / 3-fold inner nested GridSearchCV
> experiment reported in the paper. Metrics will differ from Table I in the manuscript.

**Dataset:** 1,786 Sentinel-2 observations · 99 sampling locations · 4 provinces
· 19 crop-season windows (Kharif 2015–2024, Rabi 2016–2024)

> **⚠ Label Circularity:** `crop_stress_label` is derived from NDVI, NDRE, and EVI
> via a weighted composite (score = 0.5·NDVI + 0.3·NDRE + 0.2·EVI, tertile split) —
> the same indices used as model features. Reported accuracy measures **spectral proxy
> separability**, not independently validated crop-disease detection.
> See `docs/dataset_description.md` and `data/label_definition.json` for full details.

> **⚠ Sampling methodology:** Coordinates are district/city centroids (~1 km buffer).
> No individual farm polygons or cropland mask is applied. `crop_type` is a nominal
> primary-crop category per location, not independently verified per season.

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (GroupShuffleSplit, StratifiedGroupKFold,
                                     cross_val_score)
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, accuracy_score, f1_score)
from xgboost import XGBClassifier

from feature_engineering import FEATURE_COLS

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

## 2. Load Dataset

In [ ]:
df = pd.read_csv('../data/agrosense_crop_stress_dataset.csv')
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
# Class distribution
df['crop_stress_label'].value_counts().plot(kind='bar', color=['#4CAF50','#FF9800','#F44336'])
plt.title('Class Distribution')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()

In [ ]:
# Sampling locations per province
df.groupby('province')['location_id'].nunique().sort_values().plot(kind='barh')
plt.title('Sampling Locations per Province')
plt.xlabel('Number of Sampling Locations')
plt.tight_layout()

## 3. Feature Analysis

In [ ]:
# NDVI distribution by class
for label, grp in df.groupby('crop_stress_label'):
    grp['ndvi'].plot.kde(label=label)
plt.title('NDVI Distribution by Stress Class')
plt.xlabel('NDVI')
plt.legend()
plt.tight_layout()

In [ ]:
# Feature correlation heatmap
plt.figure(figsize=(9, 7))
sns.heatmap(df[FEATURE_COLS].corr(), annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5)
plt.title('Spectral Feature Correlation Matrix')
plt.tight_layout()

## 4. Location-Grouped Train / Test Split

Uses `GroupShuffleSplit` on `location_id` so no sampling location appears in both
train and test. This avoids temporal leakage from correlated seasonal observations
at the same centroid.

In [ ]:
le = LabelEncoder()
X = df[FEATURE_COLS].values
y = le.fit_transform(df['crop_stress_label'])
groups = df['location_id'].values
classes = le.classes_

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
g_train = groups[train_idx]

print(f'Train: {len(X_train)} obs / {len(set(g_train))} locations')
print(f'Test:  {len(X_test)} obs / {len(set(groups[test_idx]))} locations')
print(f'Location overlap (should be 0): {len(set(g_train) & set(groups[test_idx]))}')

## 5. Train All Five Classifiers (with location-grouped CV)

SVM and k-NN use a `Pipeline` with `StandardScaler` to avoid data leakage
from fitting the scaler on the full dataset. CV uses `StratifiedGroupKFold`
so groups (sampling locations) are never split across folds.

In [ ]:
classifiers = {
    'Random Forest':     RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'XGBoost':           XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                                       eval_metric='mlogloss', random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, max_depth=5,
                                                    learning_rate=0.1, random_state=42),
    'SVM (RBF)':         Pipeline([('scaler', StandardScaler()),
                                   ('model', SVC(kernel='rbf', C=10, gamma='scale'))]),
    'k-NN':              Pipeline([('scaler', StandardScaler()),
                                   ('model', KNeighborsClassifier(n_neighbors=5))]),
}

trained, results = {}, []
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    trained[name] = clf
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average='macro')
    cv_f1 = cross_val_score(clf, X_train, y_train, groups=g_train,
                            cv=cv, scoring='f1_macro').mean()
    results.append({'Algorithm': name, 'Test Accuracy': acc,
                    'Test F1 Macro': f1, 'CV F1 Macro (grouped)': cv_f1})
    print(f'{name:22s}  Acc={acc:.4f}  Test-F1={f1:.4f}  CV-F1={cv_f1:.4f}')

results_df = pd.DataFrame(results)
results_df

## 6. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, (name, clf) in zip(axes, trained.items()):
    cm = confusion_matrix(y_test, clf.predict(X_test), normalize='true')
    ConfusionMatrixDisplay(cm, display_labels=classes).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=9)
plt.suptitle('Normalised Confusion Matrices (Test Set)', y=1.02)
plt.tight_layout()

## 7. Feature Importance

In [ ]:
tree_models = {k: v for k, v in trained.items()
               if hasattr(v, 'feature_importances_')}
imp = pd.DataFrame({k: v.feature_importances_ for k, v in tree_models.items()},
                    index=FEATURE_COLS).sort_values('Random Forest', ascending=False)

imp.plot(kind='bar', figsize=(10, 4))
plt.title('Feature Importance — Tree Models\n'
          '(Caution: NDVI/NDRE/EVI also define the labels — importance is inflated)')
plt.ylabel('Importance')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()

## 8. Best Model — Per-Class Report

In [ ]:
best = trained['Random Forest']
print(classification_report(y_test, best.predict(X_test), target_names=classes))